In [0]:
from pyspark.sql.functions import col, count

# Primary keys for Silver tables
# Reviews use a composite key because review_id alone
# is not unique in this dataset.
primary_keys = {
    "customers": ["customer_id"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"]
}

tables = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "category_translation"
]

results = []

for table_name in tables:

    df = spark.table(f"workspace.silver.{table_name}")

    # Check 1: Dataset is not empty
    row_count = df.count()

    results.append((
        table_name,
        "Non-empty dataset",
        "PASS" if row_count > 0 else "FAIL",
        row_count
    ))

    # Check 2: Primary key null validation
    if table_name in primary_keys:

        keys = primary_keys[table_name]

        null_condition = None

        for key in keys:
            condition = col(key).isNull()
            null_condition = (
                condition if null_condition is None
                else null_condition | condition
            )

        null_count = df.filter(null_condition).count()

        results.append((
            table_name,
            "Null primary keys",
            "PASS" if null_count == 0 else "FAIL",
            null_count
        ))

        # Check 3: Duplicate primary key validation
        duplicate_count = (
            df.groupBy(*keys)
            .count()
            .filter(col("count") > 1)
            .count()
        )

        results.append((
            table_name,
            "Duplicate primary keys",
            "PASS" if duplicate_count == 0 else "FAIL",
            duplicate_count
        ))

# Create validation report
quality_report = spark.createDataFrame(
    results,
    ["table_name", "check_name", "status", "issue_count"]
)

display(quality_report.orderBy("table_name", "check_name"))

# Summary
total_checks = quality_report.count()

passed_checks = quality_report.filter(
    col("status") == "PASS"
).count()

failed_checks = total_checks - passed_checks

print("DATA QUALITY VALIDATION SUMMARY")
print("Total checks:", total_checks)
print("Passed checks:", passed_checks)
print("Failed checks:", failed_checks)

table_name,check_name,status,issue_count
category_translation,Duplicate primary keys,PASS,0
category_translation,Non-empty dataset,PASS,71
category_translation,Null primary keys,PASS,0
customers,Duplicate primary keys,PASS,0
customers,Non-empty dataset,PASS,99441
customers,Null primary keys,PASS,0
geolocation,Non-empty dataset,PASS,738332
order_items,Duplicate primary keys,PASS,0
order_items,Non-empty dataset,PASS,112650
order_items,Null primary keys,PASS,0


DATA QUALITY VALIDATION SUMMARY
Total checks: 25
Passed checks: 25
Failed checks: 0


In [0]:
# Display only failed validation checks
failed_results = quality_report.filter(
    col("status") == "FAIL"
)

display(failed_results)

table_name,check_name,status,issue_count


In [0]:
reviews = spark.table("workspace.silver.order_reviews")

# Inspect records with missing key values
null_reviews = reviews.filter(
    col("review_id").isNull() |
    col("order_id").isNull()
)

print("RECORDS WITH NULL KEYS:")
display(null_reviews.limit(10))

# Find duplicated composite keys
duplicate_reviews = (
    reviews
    .groupBy("review_id", "order_id")
    .count()
    .filter(col("count") > 1)
.orderBy(col("count").desc()))

print("DUPLICATED REVIEW AND ORDER COMBINATIONS:")
display(duplicate_reviews)

RECORDS WITH NULL KEYS:


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,_ingested_at,_source_file,_silver_processed_at


DUPLICATED REVIEW AND ORDER COMBINATIONS:


review_id,order_id,count


In [0]:
review_path = "/Volumes/workspace/ecommerce_lakehouse/raw_files/olist_order_reviews_dataset.csv"

reviews_test = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(review_path)
)

print("Corrected review row count:", reviews_test.count())

print("Rows with null review_id:", 
      reviews_test.filter(col("review_id").isNull()).count())

print("Rows with null order_id:", 
      reviews_test.filter(col("order_id").isNull()).count())

display(reviews_test.limit(5))

Corrected review row count: 99224
Rows with null review_id: 0
Rows with null order_id: 0


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z


In [0]:
# DATA QUALITY VALIDATION SUMMARY

total_checks = quality_report.count()

passed_checks = quality_report.filter(
    col("status") == "PASS"
).count()

failed_checks = quality_report.filter(
    col("status") == "FAIL"
).count()

print("\nDATA QUALITY VALIDATION SUMMARY")
print("--------------------------------")
print(f"Total checks: {total_checks}")
print(f"Passed checks: {passed_checks}")
print(f"Failed checks: {failed_checks}")

# Display failed checks, if any
if failed_checks > 0:
    print("\nFAILED VALIDATION CHECKS:")
    display(
        quality_report.filter(col("status") == "FAIL")
    )
else:
    print("\nALL DATA QUALITY CHECKS PASSED!")


DATA QUALITY VALIDATION SUMMARY
--------------------------------
Total checks: 25
Passed checks: 25
Failed checks: 0

ALL DATA QUALITY CHECKS PASSED!
